Install and Import Libraries

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

Create Spark Session

In [3]:
spark = SparkSession.builder \
    .appName("NYC Taxi Task2") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


Load Dataset

In [4]:
df = spark.read.parquet(
    "yellow_tripdata_2024-01.parquet",
    "yellow_tripdata_2024-02.parquet",
    "yellow_tripdata_2024-03.parquet",
    "yellow_tripdata_2024-04.parquet"
)

print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 13069067
Columns: 19


Check Missing Values

In [5]:
df.select(
    [
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ]
).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|        1160538|            0|   1160538|           1160538|           0|           0|           0|          0|    0|      0|         

Handle Missing Values

In [6]:
df = df.fillna({
    "passenger_count": 1,
    "RatecodeID": 1,
    "congestion_surcharge": 0,
    "Airport_fee": 0
})

df = df.fillna({
    "store_and_fwd_flag": "N"
})

Verify Missing Values Removed

In [7]:
df.select(
    [
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ]
).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|              0|            0|         0|                 0|           0|           0|           0|          0|    0|      0|         

Remove Invalid Records

In [8]:
df = df.filter(col("trip_distance") > 0)

df = df.filter(col("fare_amount") > 0)

df = df.filter(col("tip_amount") >= 0)

Create Trip Duration Feature

In [9]:
df = df.withColumn(
    "trip_duration",
    (
        unix_timestamp("tpep_dropoff_datetime")
        -
        unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

Remove Negative Durations

In [10]:
df = df.filter(col("trip_duration") > 0)

Create Pickup Hour

In [11]:
df = df.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

Create Pickup Day

In [12]:
df = df.withColumn(
    "pickup_day",
    dayofweek("tpep_pickup_datetime")
)

Create Weekend Feature

In [13]:
df = df.withColumn(
    "is_weekend",
    when(
        col("pickup_day").isin([1,7]),
        1
    ).otherwise(0)
)

Create Target Variable

High Tip = 1

Low Tip = 0

In [14]:
df = df.withColumn(
    "high_tip",
    when(
        col("tip_amount") >= 5,
        1
    ).otherwise(0)
)

Check Class Distribution

In [15]:
df.groupBy("high_tip").count().show()

+--------+--------+
|high_tip|   count|
+--------+--------+
|       1| 2330584|
|       0|10294404|
+--------+--------+



Encode Categorical Variable

In [16]:
payment_indexer = StringIndexer(
    inputCol="payment_type",
    outputCol="payment_type_index"
)

Select Features

In [17]:
feature_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "PULocationID",
    "DOLocationID",
    "payment_type_index",
    "pickup_hour",
    "pickup_day",
    "is_weekend",
    "trip_duration"
]

Create Feature Vector

In [18]:
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="assembled_features"
)

Standard Scaling

In [19]:
scaler = StandardScaler(
    inputCol="assembled_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

PySpark Pipeline

In [20]:
pipeline = Pipeline(
    stages=[
        payment_indexer,
        assembler,
        scaler
    ]
)

Fit Pipeline

In [21]:
pipeline_model = pipeline.fit(df)

processed_df = pipeline_model.transform(df)

View Final Dataset

In [22]:
processed_df.select(
    "features",
    "high_tip"
).show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+
|features                                                                                                                                                           |high_tip|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+
|[0.0,0.004100842932506651,0.5034445081006989,2.216200501500387,3.4413706023340342,0.0,0.0,3.051010149813017,0.0,0.14184335779122328]                               |0       |
|[0.0,0.0034699440198133207,0.4214884253866316,3.7144768968809307,0.3455769642511164,0.0,0.0,3.051010149813017,0.0,0.089737634520978]                               |0       |
|[1.2332087311761881,0.002712865324581323,0.4624664667436652,4.1046530415112805,1.0799280132847389,1.4483402856736176,0.0,3.0

Final Dataset Information

In [23]:
print("Final Rows:", processed_df.count())

Final Rows: 12624988


Save Processed Dataset

In [24]:
processed_df.select(
    "features",
    "high_tip"
).write.mode("overwrite").parquet(
    "processed_taxi_data"
)

Verify Save

In [25]:
test_df = spark.read.parquet(
    "processed_taxi_data"
)

test_df.show(5)

+--------------------+--------+
|            features|high_tip|
+--------------------+--------+
|[0.0,0.0164033717...|       1|
|[1.23320873117618...|       1|
|[1.23320873117618...|       1|
|[1.23320873117618...|       0|
|[1.23320873117618...|       0|
+--------------------+--------+
only showing top 5 rows


In [27]:
test_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- high_tip: integer (nullable = true)

